# Normalizacao e armazenamento no Google Colab

Executa a fase 3 sobre os dados brutos salvos no Google Drive e grava `processed/textos_parlamentares/v1` no mesmo diretorio de dados.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
from datetime import datetime, timezone
from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/falando_nela/data')
os.environ['FALANDO_NELA_DATA_ROOT'] = str(DATA_ROOT)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# Identifica apenas esta execucao da camada processed; nao precisa bater com os run_id brutos.
PROCESSED_RUN_ID = f"processed-textos-v1-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
OVERWRITE = False
LIMIT_RECORDS = None

# Por default, o normalizador le todos os run_id brutos encontrados em raw/ e deduplica por texto_id.
# Se quiser restringir manualmente, liste aqui os run_id brutos finais desejados, por exemplo:
# RAW_RUN_IDS = ['prod-camara-plenario-baseline_v3', 'prod-senado-plenario-baseline']
RAW_RUN_IDS = []


# Para processar tudo, deixe vazio. Para isolar problemas, use exemplos como:
# DATASETS = ['senado/plenario_discursos']
# DATASETS = ['camara/plenario_discursos', 'camara/ccjc_eventos']
DATASETS = []

print('DATA_ROOT:', DATA_ROOT)
print('PROCESSED_RUN_ID:', PROCESSED_RUN_ID)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/pedblan/falando_nela.git'
REPO_DIR = Path('/content/falando_nela')
REPO_REF = ''  # opcional: branch, tag ou commit. Deixe vazio para usar o default remoto.

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all', '--tags', '--prune'], check=True)
    if not REPO_REF:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

if REPO_REF:
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
print('Repo em:', Path.cwd())
subprocess.run(['git', 'status', '--short'], check=True)


## Conferencia de entrada

Esta etapa apenas confirma que o Drive tem `raw/` e manifests de coleta antes de gerar `processed/`.


In [ ]:
import json
from pathlib import Path

raw_root = DATA_ROOT / 'raw'
manifest_root = DATA_ROOT / 'manifests'

print('raw existe:', raw_root.exists(), raw_root)
print('manifests existe:', manifest_root.exists(), manifest_root)

recent_manifests = sorted(manifest_root.glob('*.json'), key=lambda path: path.stat().st_mtime, reverse=True)[:10]
print('manifests recentes:', len(recent_manifests))
for path in recent_manifests:
    try:
        manifest = json.loads(path.read_text(encoding='utf-8'))
    except Exception as exc:
        print(path.name, 'erro ao ler:', exc)
        continue
    print(path.name, manifest.get('source'), manifest.get('dataset'), manifest.get('status'), manifest.get('record_counts'))


## Executar normalizacao

A chamada abaixo usa a funcao Python do projeto. Se o mesmo `RUN_ID` ja existir, altere `RUN_ID` ou defina `OVERWRITE = True` na celula de parametros.


In [ ]:
import json
from processamento.normalizacao import normalize_data_root

manifest = normalize_data_root(
    DATA_ROOT,
    run_id=PROCESSED_RUN_ID,
    overwrite=OVERWRITE,
    limit_records=LIMIT_RECORDS,
    datasets=DATASETS,
    raw_run_ids=RAW_RUN_IDS,
)

print(json.dumps({
    'manifest_path': manifest['manifest_path'],
    'output_root': manifest['output_root'],
    'output_records': manifest['output_records'],
    'output_record_counts': manifest['output_record_counts'],
    'skipped_counts': manifest['skipped_counts'],
}, ensure_ascii=False, indent=2))


## Inspecionar saida

Mostra o manifest e uma pequena amostra dos JSONLs processados.


In [ ]:
import json
from pathlib import Path

manifest_path = Path(manifest['manifest_path'])
manifest_loaded = json.loads(manifest_path.read_text(encoding='utf-8'))

print('Manifest:', manifest_path)
print('Arquivos de saida:', len(manifest_loaded.get('output_files', [])))
print('Registros de saida:', manifest_loaded.get('output_records'))
print('Contagens por fonte/dataset:')
print(json.dumps(manifest_loaded.get('output_record_counts', {}), ensure_ascii=False, indent=2))
print('Descartes:')
print(json.dumps(manifest_loaded.get('skipped_counts', {}), ensure_ascii=False, indent=2))


In [ ]:
import json
from pathlib import Path

sample_files = [Path(path) for path in manifest_loaded.get('output_files', [])[:5]]
for path in sample_files:
    print()
    print('Arquivo:', path)
    if not path.exists():
        print('Nao encontrado')
        continue
    with path.open('r', encoding='utf-8') as handle:
        for index, line in enumerate(handle, start=1):
            row = json.loads(line)
            print(json.dumps({
                'texto_id': row.get('texto_id'),
                'source': row.get('source'),
                'dataset': row.get('dataset'),
                'data': row.get('data'),
                'documento_tipo': row.get('documento_tipo'),
                'unidade_analitica': row.get('unidade_analitica'),
                'texto_tamanho': row.get('texto_tamanho'),
                'raw_path': row.get('raw_path'),
            }, ensure_ascii=False, indent=2))
            if index >= 2:
                break
